# Kansas Soil Water Atlas: county FAW-days

One pass over the daily maps in `faw/`. For each of the 105 counties and every day: area-weighted mean FAW, deficit-days, surplus-days (0.5 reference) and stored plant available water (km³). Writes to `data/`:

| file | content |
|---|---|
| `counties.csv` | county index, GEOID, name, area, PAW capacity, centroid, region (west/central/east third of the state) |
| `county_daily.npz` | daily arrays (days × 105): faw, deficit, surplus, paw_km3 |

Pixels are assigned to the county that contains their centre; the few border pixels outside every polygon take the nearest county.

In [ ]:
import os, sys, json, time
ROOT = r'D:\OneDrive - Kansas State University\Research projects\kansas-1km'   # the model output (faw/, soil/, boundaries/) is read from here
SEC  = r'D:\OneDrive - Kansas State University\Research projects\kansas-soil-water-atlas'   # this repository: page, figures, code and data
DATA = os.path.join(SEC, 'data'); SUM = os.path.join(DATA, 'summaries')
os.chdir(ROOT)
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.features import rasterize
from scipy.ndimage import distance_transform_edt

REF = 0.5
wk  = pd.read_csv(os.path.join(DATA, 'weeks.csv'), parse_dates=['start', 'end'])
valid = np.load(os.path.join(DATA, 'faw_days_yearly.npz'))['valid']
with rasterio.open('faw/faw_20020101.tif') as src: shape, transform = src.shape, src.transform
lat = transform.f + transform.e * (np.arange(shape[0]) + 0.5)
area = np.cos(np.deg2rad(np.broadcast_to(lat[:, None], shape)))[valid] * (abs(transform.a) * 111.32) * (abs(transform.e) * 111.32)
with rasterio.open('soil/taw.tif') as src: taw = src.read(1)[valid]

cty = gpd.read_file('boundaries/kansas_county_bnd.geojson').sort_values('NAME').reset_index(drop=True)
ids = rasterize(((g, i + 1) for i, g in enumerate(cty.geometry)), out_shape=shape, transform=transform, fill=0, dtype='int16')
miss = valid & (ids == 0)
ids = ids[tuple(distance_transform_edt(ids == 0, return_distances=False, return_indices=True))]   # nearest county
cid = ids[valid].astype(int) - 1
NC = len(cty)
A = np.bincount(cid, weights=area, minlength=NC)                     # km2
cap = np.bincount(cid, weights=taw * area, minlength=NC) * 1e-6       # km3 at FAW = 1
cen = cty.to_crs('EPSG:5070').centroid.to_crs('EPSG:4326')
lon_c = cen.x.to_numpy()
w0, w1 = transform.c, transform.c + transform.a * shape[1]
brk = [w0 + (w1 - w0) / 3, w0 + 2 * (w1 - w0) / 3]
region = np.where(lon_c < brk[0], 'west', np.where(lon_c < brk[1], 'central', 'east'))
counties = pd.DataFrame(dict(idx=np.arange(NC), geoid=cty.GEOID, name=cty.NAME, area_km2=A.round(1), capacity_km3=cap.round(4),
                             lon=lon_c.round(4), lat=cen.y.to_numpy().round(4), region=region))
counties.to_csv(os.path.join(DATA, 'counties.csv'), index=False)
print(f'{NC} counties; {miss.sum()} border pixels assigned to the nearest county; region breaks at {brk[0]:.2f} and {brk[1]:.2f} deg; '
      f'{(region == "west").sum()} west, {(region == "central").sum()} central, {(region == "east").sum()} east')

In [ ]:
last = sorted(f for f in os.listdir('faw') if f.startswith('faw_') and f.endswith('.tif'))[-1]
dates = pd.date_range('2002-01-01', pd.Timestamp(last[4:12]))
out = {k: np.empty((len(dates), NC), 'float32') for k in ('faw', 'deficit', 'surplus', 'paw_km3')}
t0 = time.time()
for t, d in enumerate(dates):
    with rasterio.open(f'faw/faw_{d:%Y%m%d}.tif') as src:
        f = np.clip(src.read(1)[valid].astype('float32'), 0, 1)
    out['faw'][t]     = np.bincount(cid, weights=f * area, minlength=NC) / A
    out['deficit'][t] = np.bincount(cid, weights=np.clip(REF - f, 0, None) * area, minlength=NC) / A
    out['surplus'][t] = np.bincount(cid, weights=np.clip(f - REF, 0, None) * area, minlength=NC) / A
    out['paw_km3'][t] = np.bincount(cid, weights=f * taw * area, minlength=NC) * 1e-6
    if t % 1000 == 0: print(f'{d:%Y-%m-%d}  {time.time() - t0:5.0f} s', flush=True)
np.savez_compressed(os.path.join(DATA, 'county_daily.npz'), dates=dates.strftime('%Y-%m-%d').to_numpy(), **out)
print(f'done in {(time.time() - t0) / 60:.1f} min')